# 23 - Agent Governance & Responsible AI

## Scenario: Audit Logging for Compliance

When an AI agent restarts a production server autonomously, who is accountable? 
To pass compliance (SOC2, ISO), autonomous agents require extreme **Auditability**. You must be able to prove exactly *why* the agent made a decision, and exactly what prompt and context it was given.

In this notebook, we will build an Audit Logging Decorator.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Audit Logger

In [2]:
import json
import hashlib
from datetime import datetime

# Simulated immutable database
audit_database = []

def log_agent_action(incident_id: str, prompt: str, tool_name: str, tool_args: dict, reason: str):
    """Logs the exact state of the agent before a destructive action."""
    prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()
    
    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "incident_id": incident_id,
        "prompt_hash": prompt_hash,
        "action": tool_name,
        "arguments": tool_args,
        "agent_reasoning": reason,
        "status": "EXECUTED"
    }
    
    audit_database.append(audit_entry)
    print(f"🔒 [Audit Log] Action recorded with hash {prompt_hash[:8]}...")


## 2. Executing an Audited Action

In [3]:
def execute_audited_restart(service_name: str, agent_thought: str):
    print(f"🤖 [Agent] Attempting to restart {service_name}...")
    
    system_prompt_used = "You are an SRE. Restart services only if memory > 90%."
    
    # 1. Write to audit log BEFORE executing
    log_agent_action(
        incident_id="INC-509",
        prompt=system_prompt_used,
        tool_name="restart_service",
        tool_args={"service": service_name},
        reason=agent_thought
    )
    
    # 2. Execute the action
    print(f"🔧 [System] Service {service_name} restarted successfully.")

execute_audited_restart("redis-checkout", "Memory is at 98%, clearing cache to prevent OOM.")

print("\n--- SOC2 Audit Review ---")
print(json.dumps(audit_database, indent=2))


🤖 [Agent] Attempting to restart redis-checkout...
🔒 [Audit Log] Action recorded with hash a19e7994...
🔧 [System] Service redis-checkout restarted successfully.

--- SOC2 Audit Review ---
[
  {
    "timestamp": "2026-08-12T16:50:25.238299",
    "incident_id": "INC-509",
    "prompt_hash": "a19e799440a18d6c1c0e31e5980ba7ef449296cef21e0918bd563d988e0fd009",
    "action": "restart_service",
    "arguments": {
      "service": "redis-checkout"
    },
    "agent_reasoning": "Memory is at 98%, clearing cache to prevent OOM.",
    "status": "EXECUTED"
  }
]


## Checkpoint

**1. Why do we hash the prompt in the audit log?**
- A) To save database space.
- B) To ensure cryptographic proof that the exact instructions given to the agent were not altered after the fact by a malicious actor.
- C) To make the prompt execute faster.
- D) To hide the prompt from the user.
